# Imports

In [ ]:
# Data Parsing:
import pandas as pd
import warnings
from pathlib import Path
import json
import random
import requests
from datetime import datetime
import sys
import numpy as np
from urllib.parse import urlparse
from types import SimpleNamespace
import requests

# Visualizations:
import matplotlib.pyplot as plt
import seaborn as sns

# Set Module path:
PROJECT_ROOT = Path().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Custom modules:
import experiment
from experiment import load_jsonl, run_experiment
from analysis_tools import analyze_run, patch_scores_csv_for_analysis
from gpt import ScratchGPTBackend, fake_post, configure_scratch_backend

### Settings:

In [ ]:
# Cleaner visualizations:
sns.set_theme()

# For consistency:
seed = 28980
np.random.seed(seed)

# Omit warnings:
warnings.filterwarnings('ignore')

## Scratch GPT integration

In [ ]:
# Local backend to match Ollama models:
SCRATCH_GPT_NAME = 'scratch-gpt'
SCRATCH_GPT_CONFIG = {
    'weights_path': 'model_weights.pt',
    'tokenizer_dir': './hftokenizer',
    'd_model': 1024,
    'n_heads': 16,
    'layers': 32,
    'vocab_size': 10000,
    'max_seq_len': 512,
    }

SCRATCH_GPT_ENDPOINTS = {'http://localhost:11434/api/generate', 'http://127.0.0.1:11434/api/generate',}

scratch_backend = ScratchGPTBackend(SCRATCH_GPT_CONFIG)
configure_scratch_backend(scratch_backend)

requests.post = fake_post
experiment.requests.post = fake_post

# Full Run:

### Load Task Dataset:

In [ ]:
ifeval = load_jsonl('data_processed/tasks_ifeval.jsonl')
ifeval_fc = load_jsonl('data_processed/tasks_ifeval_fc.jsonl')

tasks_main = ifeval + ifeval_fc
print(f'IFEval:\t\t{len(ifeval):,}')
print(f'IFEval-FC:\t{len(ifeval_fc):,}')
print(f'Total tasks:\t{len(tasks_main):,}')

### Launch IfEval Run:

In [ ]:
%%time

rng = random.Random(seed)
ifeval_shuffle = ifeval.copy()
rng.shuffle(ifeval_shuffle)
ifeval_tasks = ifeval_shuffle

run_id_scratch = 'RUN_IFEVAL_ScratchGPT'

run_experiment(
    run_id=run_id_scratch,
    models=['qwen3:8b', 'llama3.2:3b', 'gemma3:12b', 'gpt-oss:20b', SCRATCH_GPT_NAME],
    tasks=ifeval_tasks,
    repeats=12,
    temperature=0.7,
    num_predict=256,
    progress_every=720,
    endpoint='http://localhost:11434/api/generate',
    )

### Launch IfEval-FC Run:

In [ ]:
%%time

rng = random.Random(seed)
ifeval_fc_shuffle = ifeval_fc.copy()
rng.shuffle(ifeval_fc_shuffle)
ifeval_fc_tasks = ifeval_fc_shuffle

run_id_fc_scratch = 'RUN_IFEVALFC_ScratchGPT'

run_experiment(
    run_id=run_id_fc_scratch,
    models=['qwen3:8b', 'llama3.2:3b', 'gemma3:12b', 'gpt-oss:20b', SCRATCH_GPT_NAME],
    tasks=ifeval_fc_tasks,
    repeats=12,
    temperature=0.7,
    num_predict=256,
    progress_every=720,
    endpoint='http://localhost:11434/api/generate',
    )

# Evaluate Results:

In [ ]:
# Patch the two new scratch runs before analysis
patch_scores_csv_for_analysis(run_id_scratch, "scratch-gpt")
patch_scores_csv_for_analysis(run_id_fc_scratch, "scratch-gpt")

# Analyze the existing 4-model runs plus the new scratch-GPT-only runs
res_ifeval_base = analyze_run('RUN_IFEVAL_All')
res_ifeval_fc_base = analyze_run('RUN_IFEVALFC_All')

res_ifeval_scratch = analyze_run(run_id_scratch)
res_ifeval_fc_scratch = analyze_run(run_id_fc_scratch)

res_ifeval = {
    **res_ifeval_base,
    'summary_model_topology': pd.concat(
        [res_ifeval_base['summary_model_topology'], res_ifeval_scratch['summary_model_topology']],
        ignore_index=True,
    ),
    'scores': pd.concat(
        [res_ifeval_base['scores'], res_ifeval_scratch['scores']],
        ignore_index=True,
    ),
}

res_ifeval_fc = {
    **res_ifeval_fc_base,
    'summary_model_topology': pd.concat(
        [res_ifeval_fc_base['summary_model_topology'], res_ifeval_fc_scratch['summary_model_topology']],
        ignore_index=True,
    ),
    'scores': pd.concat(
        [res_ifeval_fc_base['scores'], res_ifeval_fc_scratch['scores']],
        ignore_index=True,
    ),
}

print('IFEval summary (model x topology):')
display(res_ifeval['summary_model_topology'])

print('IFEval-FC summary (model x topology):')
display(res_ifeval_fc['summary_model_topology'])

In [ ]:
# Add run labels
ifeval_model_topo = res_ifeval['summary_model_topology'].copy()
ifeval_model_topo['dataset_eval'] = 'ifeval'

ifeval_fc_model_topo = res_ifeval_fc['summary_model_topology'].copy()
ifeval_fc_model_topo['dataset_eval'] = 'ifeval-fc'

combined_model_topo = pd.concat(
    [ifeval_model_topo, ifeval_fc_model_topo],
    ignore_index=True,
    )

display(combined_model_topo)

In [ ]:
model_summary = (
    combined_model_topo
    .groupby(['dataset_eval', 'model'], as_index=False)
    .agg(
        mean_pass_rate=('pass_rate', 'mean'),
        mean_pass_std=('pass_std', 'mean'),
        mean_json_rate=('json_rate', 'mean'),
        mean_schema_rate=('schema_rate', 'mean'),
        mean_format_rate=('format_rate', 'mean'),
        total_n=('n', 'sum'),
    )
    .sort_values(['dataset_eval', 'mean_pass_rate'], ascending=[True, False])
)

display(model_summary)

In [ ]:
topology_summary = (
    combined_model_topo
    .groupby(['dataset_eval', 'topology_id'], as_index=False)
    .agg(
        mean_pass_rate=('pass_rate', 'mean'),
        mean_pass_std=('pass_std', 'mean'),
        mean_json_rate=('json_rate', 'mean'),
        mean_schema_rate=('schema_rate', 'mean'),
        mean_format_rate=('format_rate', 'mean'),
        total_n=('n', 'sum'),
    )
    .sort_values(['dataset_eval', 'mean_pass_rate'], ascending=[True, False])
)

display(topology_summary)

In [ ]:
for ds in ['ifeval', 'ifeval-fc']:
    print(f'\nPass rate pivot for {ds}:')
    pivot_pass = (
        combined_model_topo[combined_model_topo['dataset_eval'] == ds]
        .pivot(index='topology_id', columns='model', values='pass_rate')
        .sort_index()
    )
    display(pivot_pass)

In [ ]:
print('\nIFEval-FC schema_rate pivot:')
pivot_schema_fc = (
    combined_model_topo[combined_model_topo['dataset_eval'] == 'ifeval-fc']
    .pivot(index='topology_id', columns='model', values='schema_rate')
    .sort_index()
)
display(pivot_schema_fc)

print('\nIFEval-FC format_rate pivot:')
pivot_format_fc = (
    combined_model_topo[combined_model_topo['dataset_eval'] == 'ifeval-fc']
    .pivot(index='topology_id', columns='model', values='format_rate')
    .sort_index()
)
display(pivot_format_fc)

In [ ]:
scores_ifeval = res_ifeval['scores'].copy()
scores_ifeval['run_name'] = 'IFEval'

scores_ifeval_fc = res_ifeval_fc['scores'].copy()
scores_ifeval_fc['run_name'] = 'IFEval-FC'

scores_all = pd.concat([scores_ifeval, scores_ifeval_fc], ignore_index=True)

print('Combined raw score rows:', len(scores_all))
display(scores_all.head())

In [ ]:
combined_model_topo.to_csv('runs/combined_model_topology_ifeval_and_fc.csv', index=False)
model_summary.to_csv('runs/combined_model_summary_ifeval_and_fc.csv', index=False)
topology_summary.to_csv('runs/combined_topology_summary_ifeval_and_fc.csv', index=False)
scores_all.to_csv('runs/combined_scores_ifeval_and_fc.csv', index=False)

In [ ]:
fc_check = combined_model_topo[combined_model_topo['dataset_eval'] == 'ifeval-fc'].copy()

print('Unique pass_rate values (IFEval-FC):', sorted(fc_check['pass_rate'].dropna().unique())[:20], '...')
print('Unique schema_rate values (IFEval-FC):', sorted(fc_check['schema_rate'].dropna().unique())[:20], '...')
print('Unique format_rate values (IFEval-FC):', sorted(fc_check['format_rate'].dropna().unique())[:20], '...')

## Visualizations:

In [ ]:
df = pd.read_csv('./runs/combined_model_topology_ifeval_and_fc.csv')
fc = df[df['dataset_eval'] == 'ifeval-fc'].copy()

pivot = fc.pivot(index='model', columns='topology_id', values='pass_rate')

# Order rows/columns by average success so the structure is easier to read
row_order = pivot.mean(axis=1).sort_values(ascending=False).index
col_order = pivot.mean(axis=0).sort_values(ascending=False).index
pivot = pivot.loc[row_order, col_order]

plt.figure(figsize=(12, 6))
ax = sns.heatmap(
    pivot,
    annot=True,
    fmt='.4f',
    cmap='YlGnBu',
    linewidths=0.5
)

ax.set_title('IFEval-FC Pass Rate Heatmap: Model × Topology')
ax.set_xlabel('Topology')
ax.set_ylabel('Model')

plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv('./runs/combined_model_topology_ifeval_and_fc.csv')
fc = df[df['dataset_eval'] == 'ifeval'].copy()

pivot = fc.pivot(index='model', columns='topology_id', values='pass_rate')

# Order rows/columns by average success so the structure is easier to read
row_order = pivot.mean(axis=1).sort_values(ascending=False).index
col_order = pivot.mean(axis=0).sort_values(ascending=False).index
pivot = pivot.loc[row_order, col_order]

plt.figure(figsize=(12, 6))
ax = sns.heatmap(
    pivot,
    annot=True,
    fmt='.4f',
    cmap='YlGnBu',
    linewidths=0.5
)

ax.set_title('IFEval Pass Rate Heatmap: Model × Topology')
ax.set_xlabel('Topology')
ax.set_ylabel('Model')

plt.tight_layout()
plt.show()